In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, json, time
import numpy as np
import pandas as pd
from collections import Counter
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# =========================
# CONFIG
# =========================
RUN_TRAINING = True   # True: entrena y guarda / False: solo carga outputs
RANDOM_STATE = 42
TEST_SIZE = 0.30

TREE_METHOD_STACK = "gpu_hist"   # usa "hist" si no tienes CUDA
TREE_METHOD_BASELINE = "hist"    # baseline XGB+LAVA (normalmente CPU)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUT_DIR, f"run_{RUN_ID}")
os.makedirs(RUN_DIR, exist_ok=True)

print("RUN_DIR:", RUN_DIR)


RUN_DIR: outputs\run_20260204_165812


In [2]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def report_dict(y_true, y_pred, target_names):
    return classification_report(y_true, y_pred, target_names=target_names, output_dict=True)

def cm_list(cm):
    return cm.tolist()

def pretty_print(d):
    for k, v in d.items():
        if isinstance(v, float):
            print(f"{k}: {v:.6f}")
        else:
            print(f"{k}: {v}")

def count_standard_to_poor(y_true, y_pred):
    return int(np.sum((y_true == 1) & (y_pred == 2)))


In [3]:
DATA_PATH = "data_limpia.pkl"   # <-- ajusta si es necesario

df = pd.read_pickle(DATA_PATH)
target_col = "credit_score"

y_raw = df[target_col].copy()
X = df.drop(columns=[target_col]).copy()

class_order = ["Good", "Standard", "Poor"]
mapping = {cls: i for i, cls in enumerate(class_order)}
y = y_raw.map(mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("Shape df:", df.shape)
print("Clases en credit_score:\n", y_raw.value_counts(), "\n")
print("Mapping:", mapping)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

run_meta = {
    "run_id": RUN_ID,
    "data_path": DATA_PATH,
    "df_shape": list(df.shape),
    "class_order": class_order,
    "mapping": mapping,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "x_train_shape": list(X_train.shape),
    "x_test_shape": list(X_test.shape),
}
save_json(run_meta, os.path.join(RUN_DIR, "run_meta.json"))


Shape df: (100000, 29)
Clases en credit_score:
 credit_score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64 

Mapping: {'Good': 0, 'Standard': 1, 'Poor': 2}
X_train: (70000, 28) X_test: (30000, 28)


In [4]:
num_cols = [
    'age', 'monthly_inhand_salary',
    'num_bank_accounts', 'num_credit_card', 'interest_rate',
    'delay_from_due_date', 'num_of_delayed_payment', 'changed_credit_limit',
    'num_credit_inquiries', 'outstanding_debt', 'credit_utilization_ratio',
    'credit_history_age', 'total_emi_per_month', 'amount_invested_monthly',
    'monthly_balance'
]

cat_cols = [
    'occupation', 'credit_mix',
    'payment_of_min_amount', 'payment_behaviour'
]

print("num_cols:", len(num_cols))
print("cat_cols:", len(cat_cols))


num_cols: 15
cat_cols: 4


In [5]:
class LavaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, num_cols, corr_threshold=0.5, min_group_size=2, random_state=42, verbose=False):
        self.num_cols = num_cols
        self.corr_threshold = corr_threshold
        self.min_group_size = min_group_size
        self.random_state = random_state
        self.verbose = verbose

    def _build_lava_groups(self, X_num: pd.DataFrame):
        cols = X_num.columns.tolist()
        corr = X_num.corr().values
        n = len(cols)

        adj = [[False]*n for _ in range(n)]
        for i in range(n):
            for j in range(i+1, n):
                if abs(corr[i, j]) >= self.corr_threshold:
                    adj[i][j] = adj[j][i] = True

        visited = [False]*n
        groups = []
        for i in range(n):
            if not visited[i]:
                stack = [i]
                comp = []
                visited[i] = True
                while stack:
                    u = stack.pop()
                    comp.append(u)
                    for v in range(n):
                        if adj[u][v] and not visited[v]:
                            visited[v] = True
                            stack.append(v)
                if len(comp) >= self.min_group_size:
                    groups.append([cols[idx] for idx in comp])
        return groups

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("LavaTransformer espera DataFrame con nombres de columna.")

        X_num = X[self.num_cols].copy()
        self.groups_ = self._build_lava_groups(X_num)
        self.scalers_ = []
        self.pcas_ = []

        for group in self.groups_:
            scaler = StandardScaler()
            pca = PCA(n_components=1, random_state=self.random_state)
            Xg = X_num[group].values
            Xg_scaled = scaler.fit_transform(Xg)
            pca.fit(Xg_scaled)
            self.scalers_.append(scaler)
            self.pcas_.append(pca)

        self.n_groups_ = len(self.groups_)
        if self.verbose:
            print(f"[LAVA] grupos encontrados: {self.n_groups_}")
            for i, g in enumerate(self.groups_):
                print(f"  Grupo {i}: {g}")
            print()
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("LavaTransformer espera DataFrame con nombres de columna.")
        X_out = X.copy()
        if not hasattr(self, "n_groups_") or self.n_groups_ == 0:
            return X_out

        latent_list = []
        for group, scaler, pca in zip(self.groups_, self.scalers_, self.pcas_):
            Xg = X_out[group].values
            Xg_scaled = scaler.transform(Xg)
            z = pca.transform(Xg_scaled)
            latent_list.append(z)

        Z = np.hstack(latent_list)
        for i in range(Z.shape[1]):
            X_out[f"lava_latent_{i}"] = Z[:, i].ravel()
        return X_out


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), selector(dtype_include=np.number)),
            ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cat_cols),
        ],
        remainder="drop",
    )

def make_base_pipeline(corr_threshold=0.5, min_group_size=2, verbose=False):
    return Pipeline(steps=[
        ("lava", LavaTransformer(num_cols, corr_threshold, min_group_size, RANDOM_STATE, verbose)),
        ("prep", make_preprocessor())
    ])


In [6]:
knn_base = KNeighborsClassifier(
    n_neighbors=9, weights="distance", leaf_size=20, p=1, n_jobs=-1
)

rf_base = RandomForestClassifier(
    class_weight="balanced",
    max_depth=None,
    max_features=0.3,
    min_samples_leaf=3,
    min_samples_split=4,
    n_estimators=130,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_base = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    random_state=RANDOM_STATE,
    tree_method=TREE_METHOD_STACK,
    n_estimators=749,
    max_depth=9,
    learning_rate=0.0802,
    subsample=0.9442,
    colsample_bytree=0.7730,
    n_jobs=-1
)

base_models = [("knn", knn_base), ("rf", rf_base), ("xgb", xgb_base)]

meta_model_base = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    random_state=RANDOM_STATE,
    tree_method=TREE_METHOD_STACK,
    n_estimators=173,
    max_depth=9,
    learning_rate=0.12964733273336593,
    subsample=0.9442,
    colsample_bytree=0.9933692563579372,
    n_jobs=-1,
    min_child_weight=1,
    reg_alpha=0.5081987767407187,
    reg_lambda=2.3916256135817635
)


In [7]:
if RUN_TRAINING:
    t_total = time.time()

    # --------- 7.1 OOF (train) ---------
    n_classes = 3
    n_train = X_train.shape[0]
    oof_probs = {name: np.zeros((n_train, n_classes)) for name, _ in base_models}

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    t_oof = time.time()
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train), 1):
        print(f"Fold {fold} ...")
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        base_pipe = make_base_pipeline(corr_threshold=0.5, min_group_size=2, verbose=False)
        base_pipe.fit(X_tr, y_tr)

        X_tr_proc = base_pipe.transform(X_tr)
        X_val_proc = base_pipe.transform(X_val)

        for name, model in base_models:
            mdl = clone(model)
            mdl.fit(X_tr_proc, y_tr)
            oof_probs[name][val_idx, :] = mdl.predict_proba(X_val_proc)

    time_oof = (time.time() - t_oof)/60
    print(f"⏱️ OOF time: {time_oof:.2f} min")

    # --------- 7.2 Fit full preprocess + base models (train full) ---------
    t_full = time.time()
    base_pipe_full = make_base_pipeline(corr_threshold=0.5, min_group_size=2, verbose=True)
    base_pipe_full.fit(X_train, y_train)

    X_train_proc = base_pipe_full.transform(X_train)
    X_test_proc  = base_pipe_full.transform(X_test)

    test_probs = {}
    for name, model in base_models:
        mdl = clone(model)
        mdl.fit(X_train_proc, y_train)
        test_probs[name] = mdl.predict_proba(X_test_proc)

    time_full = (time.time() - t_full)/60
    print(f"⏱️ Base models fit time: {time_full:.2f} min")

    # --------- 7.3 Search blend weights (using OOF meta-train) ---------
    W = [0.2, 0.3, 0.4, 0.5]
    best_w = None
    best_f1_train = -1.0
    best_meta_train = None
    best_meta_model = None

    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

    t_w = time.time()
    for w_knn in W:
        for w_rf in W:
            w_xgb = 1.0 - w_knn - w_rf
            if w_xgb <= 0 or w_xgb > 0.8:
                continue

            blend_train = w_knn*oof_probs["knn"] + w_rf*oof_probs["rf"] + w_xgb*oof_probs["xgb"]
            confidence = blend_train.max(axis=1)
            entropy = -(blend_train * np.log(blend_train + 1e-15)).sum(axis=1)

            meta_train = np.hstack([
                oof_probs["knn"], oof_probs["rf"], oof_probs["xgb"],
                blend_train,
                confidence.reshape(-1,1),
                entropy.reshape(-1,1)
            ])

            meta_model_tmp = clone(meta_model_base)
            meta_model_tmp.fit(meta_train, y_train, sample_weight=sample_weight)
            y_pred_train_tmp = meta_model_tmp.predict(meta_train)
            f1_tmp = f1_score(y_train, y_pred_train_tmp, average="macro")

            if f1_tmp > best_f1_train:
                best_f1_train = f1_tmp
                best_w = (w_knn, w_rf, w_xgb)
                best_meta_train = meta_train
                best_meta_model = meta_model_tmp

    time_w = (time.time() - t_w)/60
    print("Best weights:", best_w, "| F1 train(meta):", best_f1_train)
    print(f"⏱️ Weight search time: {time_w:.2f} min")

    w_knn, w_rf, w_xgb = best_w

    # --------- 7.4 Meta-features test ---------
    blend_test = w_knn*test_probs["knn"] + w_rf*test_probs["rf"] + w_xgb*test_probs["xgb"]
    confidence_test = blend_test.max(axis=1)
    entropy_test = -(blend_test * np.log(blend_test + 1e-15)).sum(axis=1)

    meta_test = np.hstack([
        test_probs["knn"], test_probs["rf"], test_probs["xgb"],
        blend_test,
        confidence_test.reshape(-1,1),
        entropy_test.reshape(-1,1)
    ])

    # --------- 7.5 Meta probas (OOF train + test) ---------
    proba_train_meta = best_meta_model.predict_proba(best_meta_train)
    proba_test_meta  = best_meta_model.predict_proba(meta_test)

    # Baseline no alphas
    y_pred_test_no_alpha = proba_test_meta.argmax(axis=1)
    f1_test_no_alpha = float(f1_score(y_test, y_pred_test_no_alpha, average="macro"))
    cm_test_no_alpha = confusion_matrix(y_test, y_pred_test_no_alpha)

    # --------- 7.6 Tune alphas on TRAIN (OOF) ---------
    alphas_grid = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]
    best_f1_alpha_train = -1.0
    best_alphas = (1.0, 1.0, 1.0)

    t_a = time.time()
    for a0 in alphas_grid:
        for a1 in alphas_grid:
            for a2 in alphas_grid:
                alphas = np.array([a0, a1, a2], dtype=float)
                scaled_train = proba_train_meta / alphas
                y_pred_train_scaled = scaled_train.argmax(axis=1)
                f1m = f1_score(y_train, y_pred_train_scaled, average="macro")
                if f1m > best_f1_alpha_train:
                    best_f1_alpha_train = f1m
                    best_alphas = (a0, a1, a2)

    time_a = (time.time() - t_a)/60
    print("Best alphas:", best_alphas, "| F1 train (alphas):", best_f1_alpha_train)
    print(f"⏱️ Alpha search time: {time_a:.2f} min")

    alphas = np.array(best_alphas, dtype=float)
    proba_train_final = proba_train_meta / alphas
    proba_test_final  = proba_test_meta / alphas

    # Baseline (argmax) con alphas
    y_pred_test_alpha = proba_test_final.argmax(axis=1)
    f1_test_alpha = float(f1_score(y_test, y_pred_test_alpha, average="macro"))
    cm_test_alpha = confusion_matrix(y_test, y_pred_test_alpha)

    # --------- 7.7 Gate tuneado en TRAIN OOF (opción C) y aplicado a TEST ---------
    def standard_gate(proba, t_std):
        # default: argmax entre Good y Poor
        gp = proba[:, [0, 2]]
        y_gp = gp.argmax(axis=1)     # 0->Good, 1->Poor
        y_pred = np.where(y_gp == 0, 0, 2)
        # gate
        y_pred = np.where(proba[:, 1] >= t_std, 1, y_pred)
        return y_pred

    # Baseline TRAIN OOF (para la restricción)
    y_pred_train_base = proba_train_final.argmax(axis=1)
    base_train_f1 = float(f1_score(y_train, y_pred_train_base, average="macro"))

    # Restricción tipo opción C en TRAIN OOF: permitir caída pequeña
    TOL = 0.005
    MIN_F1_TRAIN = base_train_f1 - TOL

    T_GRID = np.round(np.arange(0.30, 0.76, 0.02), 2)
    gate_grid = []
    for t in T_GRID:
        y_pred_gate_train = standard_gate(proba_train_final, t_std=t)
        f1m = float(f1_score(y_train, y_pred_gate_train, average="macro"))
        s2p = count_standard_to_poor(y_train, y_pred_gate_train)
        gate_grid.append({"t_std": t, "f1_macro_train": f1m, "std_to_poor_train": s2p})

    gate_df = pd.DataFrame(gate_grid)
    feasible = gate_df[gate_df["f1_macro_train"] >= MIN_F1_TRAIN].copy()

    if len(feasible) == 0:
        best_t_std = None
        print("No hay t_std factibles bajo la restricción. Gate NO aplicado.")
    else:
        best_row = feasible.sort_values(by=["std_to_poor_train","f1_macro_train"], ascending=[True, False]).iloc[0]
        best_t_std = float(best_row["t_std"])
        print("✅ best_t_std (tuneado en TRAIN OOF):", best_t_std)
        display(feasible.sort_values(by=["std_to_poor_train","f1_macro_train"], ascending=[True, False]).head(10))

    if best_t_std is None:
        y_pred_test_gate = None
        f1_test_gate = None
        cm_test_gate = None
    else:
        y_pred_test_gate = standard_gate(proba_test_final, t_std=best_t_std)
        f1_test_gate = float(f1_score(y_test, y_pred_test_gate, average="macro"))
        cm_test_gate = confusion_matrix(y_test, y_pred_test_gate)

    # --------- 7.8 Guardar resultados stacking (baseline y gate) ---------
    time_total = (time.time() - t_total)/60

    lava_groups_final = int(getattr(base_pipe_full.named_steps["lava"], "n_groups_", 0))

    stacking_results = {
        "model": "STACKING_LAVA",
        "weights": {"knn": w_knn, "rf": w_rf, "xgb": w_xgb},
        "alphas": {"Good": best_alphas[0], "Standard": best_alphas[1], "Poor": best_alphas[2]},
        "lava_groups_final": lava_groups_final,
        "baseline_argmax": {
            "f1_macro_test_no_alpha": f1_test_no_alpha,
            "f1_macro_test_alpha": f1_test_alpha,
            "std_to_poor_test_no_alpha": count_standard_to_poor(y_test, y_pred_test_no_alpha),
            "std_to_poor_test_alpha": count_standard_to_poor(y_test, y_pred_test_alpha),
        },
        "gate_standard": {
            "best_t_std_train_oof": best_t_std,
            "f1_macro_test_gate": f1_test_gate,
            "std_to_poor_test_gate": None if y_pred_test_gate is None else count_standard_to_poor(y_test, y_pred_test_gate)
        },
        "times_min": {
            "oof": float(time_oof),
            "fit_base_models": float(time_full),
            "weight_search": float(time_w),
            "alpha_search": float(time_a),
            "total": float(time_total),
        }
    }

    save_json(stacking_results, os.path.join(RUN_DIR, "stacking_results.json"))

    # Reports + matrices
    save_json(report_dict(y_test, y_pred_test_no_alpha, class_order),
              os.path.join(RUN_DIR, "stacking_report_test_no_alpha.json"))
    save_json(report_dict(y_test, y_pred_test_alpha, class_order),
              os.path.join(RUN_DIR, "stacking_report_test_alpha.json"))

    save_json({"cm_test_no_alpha": cm_list(cm_test_no_alpha), "cm_test_alpha": cm_list(cm_test_alpha)},
              os.path.join(RUN_DIR, "stacking_confusions.json"))

    pd.DataFrame(cm_test_no_alpha, index=class_order, columns=class_order).to_csv(os.path.join(RUN_DIR, "cm_test_no_alpha.csv"))
    pd.DataFrame(cm_test_alpha, index=class_order, columns=class_order).to_csv(os.path.join(RUN_DIR, "cm_test_alpha.csv"))

    # Gate outputs si existe
    if y_pred_test_gate is not None:
        save_json(report_dict(y_test, y_pred_test_gate, class_order),
                  os.path.join(RUN_DIR, "stacking_report_test_gate.json"))
        save_json({"cm_test_gate": cm_list(cm_test_gate)},
                  os.path.join(RUN_DIR, "stacking_confusion_gate.json"))
        pd.DataFrame(cm_test_gate, index=class_order, columns=class_order).to_csv(os.path.join(RUN_DIR, "cm_test_gate.csv"))

    # Guardar arrays útiles para gráficas/jbook
    np.save(os.path.join(RUN_DIR, "y_test.npy"), y_test.values)
    np.save(os.path.join(RUN_DIR, "y_pred_test_alpha.npy"), y_pred_test_alpha)
    np.save(os.path.join(RUN_DIR, "proba_test_final.npy"), proba_test_final)

    # Guardar modelos (opcional)
    joblib.dump({"base_pipe_full": base_pipe_full, "meta_model": best_meta_model},
                os.path.join(RUN_DIR, "stacking_models.joblib"))

    print("\n===== STACKING RESULTS (resumen) =====")
    pretty_print(stacking_results)

    print("\n===== TEST REPORT: BASELINE (alpha) =====")
    print(classification_report(y_test, y_pred_test_alpha, target_names=class_order))
    print(confusion_matrix(y_test, y_pred_test_alpha))
    print("F1 macro (baseline alpha):", f1_test_alpha)

    if y_pred_test_gate is not None:
        print("\n===== TEST REPORT: GATE (alpha + t_std) =====")
        print("t_std:", best_t_std)
        print(classification_report(y_test, y_pred_test_gate, target_names=class_order))
        print(confusion_matrix(y_test, y_pred_test_gate))
        print("F1 macro (gate):", f1_test_gate)


Fold 1 ...


  File "c:\Users\migue\miniconda3\envs\ml_venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\migue\miniconda3\envs\ml_venv\lib\subprocess.py", line 505, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\migue\miniconda3\envs\ml_venv\lib\subprocess.py", line 951, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\migue\miniconda3\envs\ml_venv\lib\subprocess.py", line 1436, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


Fold 2 ...
Fold 3 ...
Fold 4 ...
Fold 5 ...
⏱️ OOF time: 2.53 min
[LAVA] grupos encontrados: 2
  Grupo 0: ['monthly_inhand_salary', 'monthly_balance', 'amount_invested_monthly']
  Grupo 1: ['num_bank_accounts', 'num_of_delayed_payment', 'outstanding_debt', 'credit_history_age', 'num_credit_inquiries', 'delay_from_due_date', 'interest_rate', 'num_credit_card']

⏱️ Base models fit time: 0.63 min
Best weights: (0.2, 0.5, 0.30000000000000004) | F1 train(meta): 0.8542819397922067
⏱️ Weight search time: 0.89 min
Best alphas: (1.3, 0.7, 1.3) | F1 train (alphas): 0.8661117370280961
⏱️ Alpha search time: 0.09 min
✅ best_t_std (tuneado en TRAIN OOF): 0.32


,t_std,f1_macro_train,std_to_poor_train
1,0.32,0.863815,2587
2,0.34,0.865336,2802
3,0.36,0.865460,3041
4,0.38,0.866826,3213
5,0.40,0.867070,3399
6,0.42,0.867217,3571
7,0.44,0.867679,3718
8,0.46,0.867691,3870
9,0.48,0.867250,4030
10,0.50,0.866908,4157



===== STACKING RESULTS (resumen) =====
model: STACKING_LAVA
weights: {'knn': 0.2, 'rf': 0.5, 'xgb': 0.30000000000000004}
alphas: {'Good': 1.3, 'Standard': 0.7, 'Poor': 1.3}
lava_groups_final: 2
baseline_argmax: {'f1_macro_test_no_alpha': 0.7940665580516, 'f1_macro_test_alpha': 0.8004760358529576, 'std_to_poor_test_no_alpha': 2486, 'std_to_poor_test_alpha': 2115}
gate_standard: {'best_t_std_train_oof': 0.32, 'f1_macro_test_gate': 0.7839310588862571, 'std_to_poor_test_gate': 1700}
times_min: {'oof': 2.531060516834259, 'fit_base_models': 0.6334477504094441, 'weight_search': 0.8851393063863119, 'alpha_search': 0.09350574413935343, 'total': 4.15195429722468}

===== TEST REPORT: BASELINE (alpha) =====
              precision    recall  f1-score   support

        Good       0.73      0.82      0.77      5349
    Standard       0.86      0.77      0.81     15952
        Poor       0.78      0.86      0.82      8699

    accuracy                           0.81     30000
   macro avg       0.7

In [8]:
if RUN_TRAINING:
    CV_FOLDS = 3
    N_ITER = 15

    def make_xgb_lava_pipeline(cfg):
        xgb = XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=3,
            random_state=RANDOM_STATE,
            tree_method=cfg["tree_method"],
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            learning_rate=cfg["learning_rate"],
            subsample=cfg["subsample"],
            colsample_bytree=cfg["colsample_bytree"],
            reg_alpha=cfg["reg_alpha"],
            reg_lambda=cfg["reg_lambda"],
            min_child_weight=cfg.get("min_child_weight", 1),
            n_jobs=-1
        )
        return Pipeline(steps=[
            ("lava", LavaTransformer(num_cols, cfg["corr_threshold"], cfg["min_group_size"], RANDOM_STATE, verbose=False)),
            ("prep", make_preprocessor()),
            ("xgb", xgb)
        ])

    def cv_score(cfg):
        kf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        f1s = []
        lava_counts = []
        for tr_idx, va_idx in kf.split(X_train, y_train):
            X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
            y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

            pipe = make_xgb_lava_pipeline(cfg)
            pipe.fit(X_tr, y_tr)
            lava_counts.append(getattr(pipe.named_steps["lava"], "n_groups_", 0))

            yhat = pipe.predict(X_va)
            f1s.append(f1_score(y_va, yhat, average="macro"))

        return float(np.mean(f1s)), float(np.std(f1s)), float(np.mean(lava_counts))

    rng = np.random.default_rng(RANDOM_STATE)

    def sample_cfg():
        return {
            "corr_threshold": float(rng.choice([0.4, 0.5, 0.6])),
            "min_group_size": int(rng.choice([2, 3])),
            "tree_method": TREE_METHOD_BASELINE,
            "n_estimators": int(rng.choice([300, 600, 900])),
            "max_depth": int(rng.choice([6, 8, 10])),
            "learning_rate": float(rng.choice([0.03, 0.06, 0.10])),
            "subsample": float(rng.choice([0.8, 1.0])),
            "colsample_bytree": float(rng.choice([0.7, 0.9])),
            "reg_alpha": float(rng.choice([0.0, 0.5])),
            "reg_lambda": float(rng.choice([1.0, 2.5])),
            "min_child_weight": int(rng.choice([1, 3, 5]))
        }

    print("===== BASELINE: XGB+LAVA Random Search =====")
    t0 = time.time()

    best_cfg, best_mean, best_std, best_lava = None, -1, None, None
    for i in range(1, N_ITER + 1):
        cfg = sample_cfg()
        mean_f1, std_f1, mean_lava = cv_score(cfg)
        print(f"[{i:02d}/{N_ITER}] CV F1 macro: {mean_f1:.4f} ± {std_f1:.4f} | lava≈{mean_lava:.1f}")
        if mean_f1 > best_mean:
            best_cfg, best_mean, best_std, best_lava = cfg, mean_f1, std_f1, mean_lava

    search_min = (time.time() - t0)/60
    print("\n✅ Mejor config por CV:", best_cfg)
    print(f"CV F1 macro (best): {best_mean:.4f} ± {best_std:.4f} | lava≈{best_lava:.1f}")
    print(f"⏱️ Tiempo búsqueda: {search_min:.2f} min")

    print("\n===== Entrenamiento final XGB+LAVA y reporte en TEST =====")
    t1 = time.time()
    final_pipe = make_xgb_lava_pipeline(best_cfg)
    final_pipe.fit(X_train, y_train)

    lava_final = int(getattr(final_pipe.named_steps["lava"], "n_groups_", 0))
    y_pred_test = final_pipe.predict(X_test)

    f1_test = float(f1_score(y_test, y_pred_test, average="macro"))
    cm_test = confusion_matrix(y_test, y_pred_test)

    baseline_results = {
        "model": "XGB_LAVA_BASELINE",
        "best_cfg": best_cfg,
        "cv_mean_f1_macro": float(best_mean),
        "cv_std_f1_macro": float(best_std),
        "lava_groups_cv_mean": float(best_lava),
        "lava_groups_final": lava_final,
        "f1_macro_test": f1_test,
        "times_min": {"search": float(search_min), "final_fit": float((time.time()-t1)/60)}
    }

    save_json(baseline_results, os.path.join(RUN_DIR, "xgb_lava_results.json"))
    save_json(report_dict(y_test, y_pred_test, class_order),
              os.path.join(RUN_DIR, "xgb_lava_report_test.json"))
    save_json({"cm_test": cm_list(cm_test)}, os.path.join(RUN_DIR, "xgb_lava_confusion.json"))
    pd.DataFrame(cm_test, index=class_order, columns=class_order).to_csv(os.path.join(RUN_DIR, "xgb_lava_cm_test.csv"))

    print("\n===== XGB+LAVA RESULTS (resumen) =====")
    pretty_print(baseline_results)

    print("\n===== TEST REPORT: XGB+LAVA =====")
    print(classification_report(y_test, y_pred_test, target_names=class_order))
    print(cm_test)


===== BASELINE: XGB+LAVA Random Search =====
[01/15] CV F1 macro: 0.7493 ± 0.0029 | lava≈2.0
[02/15] CV F1 macro: 0.7562 ± 0.0028 | lava≈2.0
[03/15] CV F1 macro: 0.7524 ± 0.0036 | lava≈2.0
[04/15] CV F1 macro: 0.7484 ± 0.0034 | lava≈2.0
[05/15] CV F1 macro: 0.7412 ± 0.0034 | lava≈2.0
[06/15] CV F1 macro: 0.7506 ± 0.0053 | lava≈2.0
[07/15] CV F1 macro: 0.7592 ± 0.0035 | lava≈1.0
[08/15] CV F1 macro: 0.7418 ± 0.0055 | lava≈2.0
[09/15] CV F1 macro: 0.7466 ± 0.0025 | lava≈2.0
[10/15] CV F1 macro: 0.7555 ± 0.0035 | lava≈2.0
[11/15] CV F1 macro: 0.7446 ± 0.0051 | lava≈2.0
[12/15] CV F1 macro: 0.7479 ± 0.0020 | lava≈2.0
[13/15] CV F1 macro: 0.7510 ± 0.0042 | lava≈2.0
[14/15] CV F1 macro: 0.7479 ± 0.0034 | lava≈2.0
[15/15] CV F1 macro: 0.7438 ± 0.0049 | lava≈2.0

✅ Mejor config por CV: {'corr_threshold': 0.6, 'min_group_size': 3, 'tree_method': 'hist', 'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.06, 'subsample': 0.8, 'colsample_bytree': 0.9, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'm

In [9]:
def safe_read(path):
    return load_json(path) if os.path.exists(path) else None

stacking = safe_read(os.path.join(RUN_DIR, "stacking_results.json"))
baseline = safe_read(os.path.join(RUN_DIR, "xgb_lava_results.json"))

rows = []

if baseline:
    rows.append({
        "Model": "XGBoost + LAVA (tuned)",
        "F1_macro_TEST": baseline["f1_macro_test"],
        "CV_mean_F1_macro": baseline["cv_mean_f1_macro"],
        "LAVA_groups_final": baseline["lava_groups_final"],
        "Std→Poor_TEST": None
    })

if stacking:
    rows.append({
        "Model": "Stacking + LAVA (argmax, alpha)",
        "F1_macro_TEST": stacking["baseline_argmax"]["f1_macro_test_alpha"],
        "CV_mean_F1_macro": None,
        "LAVA_groups_final": stacking["lava_groups_final"],
        "Std→Poor_TEST": stacking["baseline_argmax"]["std_to_poor_test_alpha"]
    })
    if stacking["gate_standard"]["best_t_std_train_oof"] is not None:
        rows.append({
            "Model": "Stacking + LAVA + Gate(Standard)",
            "F1_macro_TEST": stacking["gate_standard"]["f1_macro_test_gate"],
            "CV_mean_F1_macro": None,
            "LAVA_groups_final": stacking["lava_groups_final"],
            "Std→Poor_TEST": stacking["gate_standard"]["std_to_poor_test_gate"]
        })

summary_df = pd.DataFrame(rows)
summary_path = os.path.join(RUN_DIR, "paper_summary_table.csv")
summary_df.to_csv(summary_path, index=False)

print("✅ Guardado:", summary_path)
display(summary_df)


✅ Guardado: outputs\run_20260204_165812\paper_summary_table.csv


,Model,F1_macro_TEST,CV_mean_F1_macro,LAVA_groups_final,Std→Poor_TEST
0,XGBoost + LAVA (tuned),0.783217,0.759227,1,NaN
1,"Stacking + LAVA (argmax, alpha)",0.800476,NaN,2,2115.0
2,Stacking + LAVA + Gate(Standard),0.783931,NaN,2,1700.0


In [10]:
# ============================================
# CARGA RÁPIDA (sin entrenar)
# ============================================
# Tu compañero:
# 1) pone RUN_TRAINING=False en Celda 1
# 2) pega el RUN_DIR correcto aquí
# 3) corre esta celda y ya tiene todo

# RUN_DIR = "outputs/run_YYYYMMDD_HHMMSS"  # <-- pega aquí el tuyo

assert os.path.exists(RUN_DIR), f"No existe RUN_DIR: {RUN_DIR}"
print("Cargando desde:", RUN_DIR)

run_meta = load_json(os.path.join(RUN_DIR, "run_meta.json"))
print("run_id:", run_meta["run_id"])
print("df_shape:", run_meta["df_shape"])
print("split:", run_meta["x_train_shape"], run_meta["x_test_shape"])

if os.path.exists(os.path.join(RUN_DIR, "stacking_results.json")):
    stacking = load_json(os.path.join(RUN_DIR, "stacking_results.json"))
    print("\n--- STACKING RESULTS ---")
    pretty_print(stacking)

if os.path.exists(os.path.join(RUN_DIR, "xgb_lava_results.json")):
    baseline = load_json(os.path.join(RUN_DIR, "xgb_lava_results.json"))
    print("\n--- XGB+LAVA RESULTS ---")
    pretty_print(baseline)

csv_path = os.path.join(RUN_DIR, "paper_summary_table.csv")
if os.path.exists(csv_path):
    print("\nTabla paper_summary_table.csv:")
    display(pd.read_csv(csv_path))


Cargando desde: outputs\run_20260204_165812
run_id: 20260204_165812
df_shape: [100000, 29]
split: [70000, 28] [30000, 28]

--- STACKING RESULTS ---
model: STACKING_LAVA
weights: {'knn': 0.2, 'rf': 0.5, 'xgb': 0.30000000000000004}
alphas: {'Good': 1.3, 'Standard': 0.7, 'Poor': 1.3}
lava_groups_final: 2
baseline_argmax: {'f1_macro_test_no_alpha': 0.7940665580516, 'f1_macro_test_alpha': 0.8004760358529576, 'std_to_poor_test_no_alpha': 2486, 'std_to_poor_test_alpha': 2115}
gate_standard: {'best_t_std_train_oof': 0.32, 'f1_macro_test_gate': 0.7839310588862571, 'std_to_poor_test_gate': 1700}
times_min: {'oof': 2.531060516834259, 'fit_base_models': 0.6334477504094441, 'weight_search': 0.8851393063863119, 'alpha_search': 0.09350574413935343, 'total': 4.15195429722468}

--- XGB+LAVA RESULTS ---
model: XGB_LAVA_BASELINE
best_cfg: {'corr_threshold': 0.6, 'min_group_size': 3, 'tree_method': 'hist', 'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.06, 'subsample': 0.8, 'colsample_bytree': 0

,Model,F1_macro_TEST,CV_mean_F1_macro,LAVA_groups_final,Std→Poor_TEST
0,XGBoost + LAVA (tuned),0.783217,0.759227,1,NaN
1,"Stacking + LAVA (argmax, alpha)",0.800476,NaN,2,2115.0
2,Stacking + LAVA + Gate(Standard),0.783931,NaN,2,1700.0
